# Resumen automático — Extractivo vs Abstractivo con Transformers (**Student Version**)

**Task:** Automatic text summarization. Compare a simple **extractive** method with a **Transformer-based abstractive** model.

**Models / tools:** 
- `sshleifer/distilbart-cnn-12-6` (distilled BART, abstractive summarization).
- Una implementación propia sencilla para resumen extractivo (frecuencia de palabras + selección de frases).

---

## Objetivos del notebook

En este notebook vas a:

1. Cargar un modelo de resumen **abstractivo** preentrenado usando `transformers`.
2. Implementar un pequeño sistema de resumen **extractivo** basado en puntuación de frases.
3. Comparar ambos enfoques (extractivo vs abstractivo) sobre varios textos.
4. Implementar una mini-métrica basada en **solapamiento de palabras** (aprox. tipo ROUGE-1 muy simplificado).
5. Analizar ventajas, limitaciones y errores típicos de cada enfoque.

---

## Qué se asume que ya sabes

- Diferencia conceptual entre **resumen extractivo** y **abstractivo** (Tema 4).
- Conceptos básicos de Transformers encoder–decoder.
- Python y nociones de librerías de PLN.


## 1) (Opcional) Instalación / actualización de librerías

**Qué hacer:**  
Si estás en Google Colab o en un entorno donde no tienes las librerías instaladas (o quieres actualizar), puedes usar la celda de abajo.

- Si ya tienes el entorno preparado (p. ej. proporcionado por el profesor), puedes **saltar esta celda**.


In [ ]:
# TODO (opcional): instala o actualiza las librerías necesarias.
# Ejemplo orientativo (NO lo dejes comentado si lo necesitas):
# !pip install -q "transformers>=4.40.0" datasets nltk

## 2) Imports y configuración básica

**Qué hacer:**

1. Importar:
   - `pipeline` de `transformers` (lo usaremos para el resumen abstractivo).
   - `numpy` y `re` para pequeñas utilidades.
   - `nltk` para segmentar frases (o, si prefieres, algún método propio sencillo).
2. Descargar, si es necesario, los recursos de tokenización de `nltk` (por ejemplo, `punkt`).
3. Comprobar que todo se importa sin errores.


In [ ]:
# TODO: importa las librerías necesarias.
# Pistas:
# from transformers import pipeline
# import numpy as np
# import re
# import nltk
#
# TODO: si usas nltk, descarga los recursos necesarios (por ejemplo, nltk.download("punkt")).


## 3) Carga del modelo de resumen abstractivo (Transformers)

Vamos a usar un modelo de resumen abstractivo preentrenado.  
Para esta práctica utilizaremos `sshleifer/distilbart-cnn-12-6` (una versión "distilled" de BART entrenada para resumen de noticias en inglés).

**Qué hacer:**

1. Crear un `pipeline` de tipo `"summarization"` usando el modelo indicado.
2. Guardarlo en una variable (por ejemplo, `summarizer`).
3. Probarlo con un texto corto para verificar que funciona.

> Nota: el modelo está entrenado principalmente para inglés. Puedes usar textos en inglés y, si quieres, experimentar con textos en español (aunque la calidad puede bajar).


In [ ]:
# TODO: crea un pipeline de resumen abstractivo.
# Pistas:
# summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
#
# TODO: define un pequeño texto de prueba (en inglés) y genera un resumen básico
# usando el pipeline. Imprime el texto original y el resumen.


## 4) Definición de un pequeño conjunto de documentos

Trabajaremos con un pequeño conjunto de textos para resumir (por ejemplo, fragmentos de noticias, artículos de blog o descripciones técnicas).

**Qué hacer:**

1. Define una lista `documents` con varios textos (mínimo 3–4).
   - Pueden ser textos cortos/medios (p.ej. 3–8 frases cada uno).
   - Idealmente en inglés, para que el modelo de BART funcione razonablemente bien.
2. Imprime el número de documentos y, opcionalmente, la longitud (número de palabras) de cada uno.

Puedes empezar con un texto dado por el profesor y añadir otros que te parezcan interesantes.


In [ ]:
# TODO: define una lista `documents` con varios textos a resumir.

# Ejemplo de estructura (puedes adaptarlo o reemplazarlo):
# documents = [
#     """Natural language processing (NLP) is a subfield of artificial intelligence
#     that focuses on the interaction between computers and humans through natural language.
#     The ultimate objective of NLP is to read, decipher, understand, and make sense of
#     human language in a valuable way.""".strip(),
#     ...
# ]
#
# TODO: imprime cuántos documentos hay y, por ejemplo, la longitud aproximada de cada uno.


## 5) Función auxiliar de resumen abstractivo

Para reutilizar el modelo de forma cómoda, vamos a definir una función que:

- Reciba un texto.
- Llame al `summarizer` con parámetros como `max_length`, `min_length`, `num_beams`.
- Devuelva el resumen como string.

**Qué hacer:**

1. Define una función `summarize_abstractively(text, max_length=80, min_length=20, num_beams=4)`.
2. Dentro, llama al `pipeline` de resumen y extrae el campo `"summary_text"`.
3. Prueba la función con uno de los documentos de tu lista.


In [ ]:
# TODO: implementa la función `summarize_abstractively` que envuelva al pipeline.
# Requisitos mínimos:
# - Debe aceptar un string `text`.
# - Debe permitir ajustar `max_length`, `min_length` y `num_beams`.
# - Debe devolver el resumen como string (no como lista de diccionarios).
#
# TODO: prueba la función con al menos un documento de `documents`.


## 6) Implementación de un resumen extractivo sencillo

Ahora vamos a implementar un resumen **extractivo** muy simple basado en frecuencia de palabras:

Idea básica:
1. Segmentar el texto en frases.
2. Calcular la frecuencia de cada palabra (ignorando palabras muy frecuentes, si quieres).
3. Asignar a cada frase una puntuación que sea la suma (o media) de las frecuencias de sus palabras.
4. Ordenar las frases por puntuación y seleccionar las mejores (según un ratio de compresión).

**Qué hacer:**

1. Implementa una función `split_sentences(text)` que devuelva una lista de frases.
   - Puedes usar `nltk.sent_tokenize` o un método simplificado (p.ej. dividir por puntos, con cuidado).
2. Implementa una función `extractive_summary(text, ratio=0.3)` que:
   - Use `split_sentences`.
   - Calcule frecuencias de palabras en todo el texto.
   - Asigne una puntuación a cada frase.
   - Seleccione aproximadamente el `ratio` de las frases (por ejemplo, 30%).

Al final, la función debe devolver un string con las frases seleccionadas en el **orden original**.


In [ ]:
# TODO: implementa `split_sentences(text)` y `extractive_summary(text, ratio=0.3)`.
# Pistas:
# - Para `split_sentences`, puedes usar nltk.sent_tokenize(text).
# - Para las palabras, puedes pasar todo a minúsculas, eliminar signos de puntuación con `re`
#   y dividir por espacios.
# - Para seleccionar frases, puedes ordenar por puntuación pero mantener luego el orden original
#   de las frases seleccionadas cuando construyas el resumen final.


## 7) Comparación cualitativa: extractivo vs abstractivo

Vamos a aplicar ambos sistemas (extractivo y abstractivo) a cada documento y ver los resultados.

**Qué hacer:**

1. Para cada documento en `documents`:
   - Imprime (si quieres, sólo las primeras frases) del documento original.
   - Genera y muestra el resumen **abstractivo** con tu función `summarize_abstractively`.
   - Genera y muestra el resumen **extractivo** con tu función `extractive_summary`.
2. Observa:
   - ¿Cuánto se reduce la longitud?
   - ¿Qué tipo de información se mantiene o se pierde?
   - ¿Te parece más legible uno u otro?

Escribe tus observaciones en una celda de texto aparte.


In [ ]:
# TODO: recorre `documents` y muestra, para cada uno,
# el documento original (quizá truncado), el resumen abstractivo y el extractivo.
#
# Pistas:
# - Puedes limitar la impresión del documento original a las primeras N palabras o frases
#   para que no ocupe demasiado.


## 8) Mini-métrica tipo ROUGE-1 (muy simplificada)

Vamos a implementar una métrica muy simple inspirada en ROUGE-1:

- Tokenizamos el texto original (`ref`) y el resumen (`hyp`) en palabras.
- Calculamos qué porcentaje de palabras **distintas** del documento original aparecen también en el resumen.

Esto NO es ROUGE real, pero sirve para hacerse una idea de **cobertura léxica**.

**Qué hacer:**

1. Implementa una función `simple_rouge1(hyp, ref)` que:
   - Tokenice ambos textos (con una tokenización sencilla).
   - Calcule el conjunto de palabras en `ref` y en `hyp`.
   - Devuelva `overlap / len(ref_set)` donde `overlap` es el tamaño de la intersección de ambos conjuntos.
2. Aplica esta función para comparar:
   - Resumen abstractivo vs documento original.
   - Resumen extractivo vs documento original.

y observa qué valores obtienes.


In [ ]:
# TODO: implementa `simple_rouge1(hyp, ref)`.
# Requisitos mínimos:
# - Convierte a minúsculas y elimina signos de puntuación simples.
# - Usa conjuntos de palabras para calcular el solapamiento.
#
# TODO: prueba la función con un ejemplo sencillo para comprobar que funciona.


## 9) Evaluación básica: ROUGE-1 simple para todos los documentos

Vamos a calcular la mini-métrica para todos los documentos y comparar extractivo vs abstractivo.

**Qué hacer:**

1. Para cada documento `doc` en `documents`:
   - Genera un resumen abstractivo (`abs_sum`).
   - Genera un resumen extractivo (`ext_sum`).
   - Calcula:
     - `r_abs = simple_rouge1(abs_sum, doc)`
     - `r_ext = simple_rouge1(ext_sum, doc)`
2. Guarda los resultados en alguna estructura (lista, `numpy.array`, etc.).
3. Calcula los promedios de `r_abs` y `r_ext`.
4. Imprime los resultados por documento y los promedios globales.

**Mini-análisis (en markdown):**  
¿La métrica favorece sistemáticamente al resumen extractivo? ¿Coincide con tu percepción de calidad? ¿Qué limitaciones ves en esta métrica?


In [ ]:
# TODO: implementa el bucle de evaluación que calcule simple_rouge1
# para resúmenes abstractivos y extractivos en todos los documentos.
#
# Pistas:
# - Puedes guardar tu modelo de abstracción y tu función extractiva tal como están.
# - Esta celda llamará de nuevo a las funciones, así que puede tardar un poco;
#   si quieres ahorrar tiempo, puedes reutilizar resúmenes precomputados en una lista.


## 10) Factualidad y alucinaciones en resumen abstractivo

Los modelos de resumen abstractivo pueden:

- Introducir detalles **no presentes** en el texto original (alucinaciones).
- Cambiar matices importantes.
- Omitir información crítica.

**Qué hacer:**

1. Intenta diseñar o buscar un documento donde el modelo abstractivo:
   - Añada algún detalle que NO está en el texto original, o
   - Cambie un dato importante (fecha, número, nombre, etc.).
2. Analiza el caso en una celda de texto:
   - Copia el documento original.
   - Copia el resumen abstractivo generado.
   - Señala qué parte es incorrecta o inventada.
   - Explica por qué esto puede ser peligroso en aplicaciones reales.

Relaciona esto con lo visto en la teoría sobre **factualidad** y **alucinaciones** en modelos generativos.


## 11) Conclusiones

En esta sección debes escribir un breve resumen (5–10 líneas) con tus conclusiones sobre:

- Diferencias observadas entre resumen extractivo y abstractivo.
- Cuando te ha parecido mejor uno u otro (según el tipo de texto).
- Qué te dice la mini-métrica tipo ROUGE-1 sobre la cobertura léxica.
- Cómo se relacionan tus observaciones con los apartados 4.6–4.8 del Tema 4.

Puedes comentar también ideas de extensiones: probar otros modelos, otros idiomas, ajustar hiperparámetros, etc.
